# 1-Click Bangla-Gemma to LiteRT (.litertlm / .task) Exporter for Android
This notebook converts **`emonHasan01/bangla-gemma-pro`** (or any fine-tuned Gemma/Bangla model from Hugging Face) into an on-device `.litertlm` / `.task` container using Google's official `litert-torch export_hf` tool.

Once exported, you can download the file and move it into your Android project's `app/src/main/assets/` directory.

### Step 1: Install LiteRT-Torch and Google AI Edge Tools

In [ ]:
!pip install --upgrade pip
!pip install litert-torch litert-lm-builder transformers huggingface_hub sentencepiece protobuf accelerate

### Step 2: Run the Official Google AI Edge `export_hf` Converter
This command automatically:
1. Pulls the model & tokenizer from Hugging Face (`emonHasan01/bangla-gemma-pro`).
2. Traces the autoregressive causal LM graph with KV cache.
3. Quantizes weights (INT8 dynamic / INT4) to fit within mobile RAM.
4. Packages weights, SentencePiece tokenizer, and Gemma chat templates into a `.litertlm` container.

In [ ]:
import os
os.makedirs("/content/output", exist_ok=True)

# Convert directly from Hugging Face to LiteRT container
!litert-torch export_hf \
  --model=emonHasan01/bangla-gemma-pro \
  --output_dir=/content/output \
  --bundle_litert_lm=True \
  --externalize_embedder=True

### Step 3: Verify the Exported Artifacts

In [ ]:
import os

print("Exported files in /content/output:")
for filename in sorted(os.listdir("/content/output")):
    filepath = os.path.join("/content/output", filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  -> {filename} ({size_mb:.1f} MB)")

### Step 4: Download the Compiled Model to Your Computer
Downloads the `.litertlm` (or `.task`) file. Move it to `app/src/main/assets/`.

In [ ]:
import glob
from google.colab import files

candidates = glob.glob("/content/output/*.litertlm") + glob.glob("/content/output/*.task")
if candidates:
    selected = candidates[0]
    print(f"Downloading model bundle: {selected}")
    files.download(selected)
else:
    # Fallback to downloading all output files
    for f in glob.glob("/content/output/*"):
        files.download(f)

print("Done! Copy the downloaded file to your project: app/src/main/assets/")